# Notebook 2: Phan tich Van ban

Phan tich tu khoa, tan suat va do dai bai viet.

In [ ]:
import sys, re
from pathlib import Path
from collections import Counter
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.storage import DataStorage
from config.settings import DB_PATH, RAW_DIR

plt.style.use('seaborn-v0_8-darkgrid')
storage = DataStorage(db_path=DB_PATH, raw_dir=RAW_DIR)
df = storage.load_from_sqlite()
print(f'Tai {len(df)} bai viet')

## 1. Lam sach van ban

In [ ]:
STOPWORDS = set("""
    va cua la co trong voi de tu cac mot da duoc nay nhung
    cho nhu khong ve khi sau cung do boi tai nhung day thi
    rang theo ma vi hay lai vao ra chi nhieu hon dang se
    nen den qua tren duoi truoc con len can phai nguoi minh
    ho chung chinh lan tai sao ong ba anh chi the a an in
    to of and for is are was were with this that
""".split())

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return text

def tokenize(text):
    return [
        w for w in re.findall(r'\b[a-zA-Z\u00C0-\u1EF9]{3,}\b', clean_text(text))
        if w not in STOPWORDS
    ]

df['tokens_title'] = df['title'].apply(tokenize)
df['tokens_content'] = df['content'].apply(tokenize)
print('Tokenize xong')
print('Vi du tokens tieu de:', df['tokens_title'].iloc[0][:10])

## 2. Top 20 tu khoa trong tieu de

In [ ]:
all_title_words = [w for tokens in df['tokens_title'] for w in tokens]
title_counter = Counter(all_title_words)
top_title = dict(title_counter.most_common(20))

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('viridis', 20)
bars = ax.barh(list(top_title.keys())[::-1], list(top_title.values())[::-1], color=colors)
ax.set_title('Top 20 tu khoa trong tieu de', fontsize=14, fontweight='bold')
ax.set_xlabel('Tan suat')
for bar, val in zip(bars, list(top_title.values())[::-1]):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Top 20 tu khoa trong noi dung

In [ ]:
all_content_words = [w for tokens in df['tokens_content'] for w in tokens]
content_counter = Counter(all_content_words)
top_content = dict(content_counter.most_common(20))

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(list(top_content.keys())[::-1], list(top_content.values())[::-1],
               color=sns.color_palette('plasma', 20))
ax.set_title('Top 20 tu khoa trong noi dung', fontsize=14, fontweight='bold')
ax.set_xlabel('Tan suat')
plt.tight_layout()
plt.show()

## 4. So sanh tu khoa giua cac nguon

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, source in zip(axes, df['source'].unique()):
    subset = df[df['source'] == source]
    words = [w for tokens in subset['tokens_title'] for w in tokens]
    top = dict(Counter(words).most_common(15))
    ax.barh(list(top.keys())[::-1], list(top.values())[::-1],
            color='#667eea' if source == 'vnexpress' else '#764ba2')
    ax.set_title(f'Top 15 tu khoa - {source.upper()}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Tan suat')

plt.tight_layout()
plt.show()

## 5. Phan tich do dai tieu de

In [ ]:
df['title_length'] = df['title'].apply(lambda x: len(str(x).split()))

print('=== THONG KE DO DAI TIEU DE (so tu) ===')
print(df.groupby('source')['title_length'].describe().round(2))

fig, ax = plt.subplots(figsize=(10, 5))
for source in df['source'].unique():
    subset = df[df['source'] == source]['title_length']
    ax.hist(subset, bins=20, alpha=0.7, label=source)
ax.set_title('Phan phoi do dai tieu de (so tu)', fontsize=13, fontweight='bold')
ax.set_xlabel('So tu trong tieu de')
ax.set_ylabel('So bai')
ax.legend()
plt.tight_layout()
plt.show()